# 23 · Контроль: почему h=6 в sweep (PR24 0.099) отличается от §9 (0.421)?

Сомнение: «точно ли прогоны правильные?» Три причинных гипотезы:
(A) разный способ построения события — §9 делает «серьёзным» только сам
задымившийся канал при подтверждении объекта, а sweep-эпизод — ВСЕ каналы
объекта в бакете старта;
(B) разный протокол обучения — §9 CPU/15000/ES300 vs sweep GPU/12000/ES250;
(C) баг прогона (тогда §9 и sweep несопоставимы вовсе).

План: один протокол (GPU, iters=900, lr=0.03, depth=10, ES=250,
sample 12000/8000/8000, holdout seed777), три события:
1) §9-колонка «серьёзных»;
2) 6ч эпизод, только дымовые каналы («серьёзных_6эп & задымлений>0»);
3) 6ч эпизод, все каналы (для справки — 0.099 из sweep).


In [1]:
%matplotlib inline
import sys
import pathlib

_HERE = pathlib.Path.cwd()
RESEARCH = _HERE.parent if _HERE.name == "notebooks" else _HERE
if str(RESEARCH) not in sys.path:
    sys.path.insert(0, str(RESEARCH))

import numpy as np
import pandas as pd

from catboost import CatBoostClassifier, Pool

import data_utils as du
import features as fe
import tte_pipeline as tte
from tte_experiments import survival_metrics, pr_at_horizon


DATASET = du.find_dataset_dir()
hour_ns = fe.BUCKET_HOURS * 3600 * 10 ** 9
EV = pd.read_parquet(DATASET / "fire_raw_events_30m.parquet")

panel = pd.read_csv(DATASET / "subdaily_panel_fire6h.csv",
                    dtype={"ид_канала_данных": str, "ид_объект": str})
panel = tte.refit_z_train_only(panel)
print("панель:", panel.shape, "| база §9-события: {:.3f}%".format(
    panel["серьёзных"].mean() * 100))


панель: (1204508, 143) | база §9-события: 8.522%


In [2]:
# Строим 6ч эпизод (как в sweep), затем дымовый сабсет
GAP6 = 6 * 3600 * 10 ** 9
def episodes6():
    out = []
    for об, grp in EV.groupby("об"):
        g = grp.sort_values("t")
        t = g["t"].to_numpy()
        gid = np.cumsum(np.concatenate([[True], t[1:] - t[:-1] > GAP6])) - 1
        g2 = g.copy(); g2["gid"] = gid
        cnt = g2[g2["тип"] == "дым"].groupby("gid")["ид"].nunique().rename("n_dym")
        agg = g2.groupby("gid").agg(
            t0=("t", "min"), n_manual=("тип", lambda s: int((s == "ручной").sum())))
        agg = agg.join(cnt).fillna({"n_dym": 0})
        agg = agg[(agg["n_dym"] >= 2) | (agg["n_manual"] > 0)]
        d = agg.reset_index(); d["об"] = об
        out.append(d)
    return pd.concat(out, ignore_index=True)

EP6 = episodes6()
EP6["бакет"] = (EP6["t0"] // hour_ns).astype(np.int64)
start_map = EP6[["об", "бакет"]].drop_duplicates().rename(columns={"об": "ид_объект"})

pn = panel.copy()
pn = pn.merge(start_map, on=["ид_объект", "бакет"], how="left", indicator="m")
pn["серьёзных_6эп"] = (pn["m"] == "both").astype(np.int8)
pn["серьёзных_6эп_дым"] = pn["серьёзных_6эп"] & (pn["задымлений"].to_numpy() > 0)
pn = pn.drop(columns=["m"])
print("база: 6эп все = {:.3f}% | 6эп дым. = {:.3f}% | §9 = {:.3f}%".format(
    pn["серьёзных_6эп"].mean() * 100, pn["серьёзных_6эп_дым"].mean() * 100,
    panel["серьёзных"].mean() * 100))


база: 6эп все = 9.801% | 6эп дым. = 6.219% | §9 = 8.522%


In [3]:
def run(col):
    p = tte.add_series_context(pn.copy(), event_col=col)
    s = tte.add_tte_labels(p)
    s["ид_объект_code"] = s["ид_объект"].astype("category").cat.codes
    s["тип_датчика_code"] = s["тип_датчика"].astype("category").cat.codes
    tr, va, ho = tte.split_subjects(s)
    Xc = tte.subject_features(tr)
    trs = tte.sample_subjects(tr, 12000, 8000, 8000, seed=42)
    vas = tte.sample_subjects(va, 2000, 2000, 2000, seed=43)
    hos = tte.sample_holdout(ho, 20000, seed=777)
    X_tr, y_tr = tte.expand_person_time(trs, Xc, tte.HORIZON_BUCKETS)
    X_va, y_va = tte.expand_person_time(vas, Xc, tte.HORIZON_BUCKETS)
    m = CatBoostClassifier(iterations=900, learning_rate=0.03, depth=10,
                           random_seed=42, task_type="GPU",
                           loss_function="Logloss", eval_metric="Logloss",
                           early_stopping_rounds=250, verbose=0,
                           allow_writing_files=False)
    m.fit(Pool(X_tr, y_tr), eval_set=Pool(X_va, y_va))
    surviv_tr = tte.make_surv_struct(trs["event_flag"], trs["obs_days"])
    surviv_ho = tte.make_surv_struct(hos["event_flag"], hos["obs_days"])
    Xh, _ = tte.expand_person_time(hos, Xc, tte.HORIZON_BUCKETS, fixed_horizon=True)
    S = np.cumprod(1 - m.predict_proba(Xh)[:, 1].reshape(-1, tte.HORIZON_BUCKETS), axis=1)
    met = survival_metrics(surviv_tr, surviv_ho, 1 - S[:, -1], S, tte.EVAL_TIMES)
    y24 = ((hos["event_flag"] == 1) & (hos["obs_days"] <= 1.01)).to_numpy(int)
    p24 = 1 - S[:, int(round(1.0 * tte.STEPS_PER_DAY)) - 1]
    pr = pr_at_horizon(y24, p24)
    return met, pr

for name, col in [("§9 событие (серьёзных)", "серьёзных"),
                  ("6ч эпизод, дым.", "серьёзных_6эп_дым"),
                  ("6ч эпизод, все", "серьёзных_6эп")]:
    met, pr = run(col)
    print(f"[{name}] Uno-C={met['uno_c']:.3f} | PR-AUC(24ч)={pr['pr_auc']:.3f} "
          f"(база {pr['base']*100:.2f}%)", flush=True)


[§9 событие (серьёзных)] Uno-C=0.826 | PR-AUC(24ч)=0.443 (база 1.53%)


[6ч эпизод, дым.] Uno-C=0.784 | PR-AUC(24ч)=0.065 (база 1.01%)


[6ч эпизод, все] Uno-C=0.660 | PR-AUC(24ч)=0.115 (база 3.12%)


### Интерпретация
- «§9 событие» при sweep-протоколе ≈0.4 → оба замера корректны, расхождение —
  из-за способа построения таргета, не из-за бага/протокола.
- «6ч эпизод, дым.» заметно лучше «6ч эпизод, все» → подтверждение, что
  размазывание события на не-дымовые каналы вредит модели.
